In [1]:
import time
from pop import Pilot

Car = Pilot.AutoCar()

In [8]:
import subprocess as sp, time
from IPython.display import display, Javascript
from ipywidgets import widgets
from pop import Util

# (SMBus, Address, Name)
devs=(('8','68','6축 센서'),
      ('1','5c','휠/카메라 모터 드라이버'),
      ('1','5e','전/후진 모터 드라이버'),
      ('8','0a','오디오 드라이버'))

print("- 감지된 장치 -")
det=True

for line, dev, name in devs:
    try:
        num=str(int(dev,16))
        ret=sp.check_output(['i2cdetect -y -r '+line+' '+num+' '+num+' | grep -oP "UU|'+dev+'"'], shell=True).decode('UTF-8')
        if dev in ret or 'UU' in ret:
            print("[O] "+name)
    except:
        det=False
        print("[X] "+name)
            
Camera, Cds, Pilot = (None, None, None)

if not det: 
    print("\n감지되지 않은 장치가 있습니다.\n 확인 후 테스트를 진행해주세요.") 
else:
    from pop import Camera as cam, Cds as cds, Pilot as pilot, LiDAR
    Camera, Cds, Pilot = (cam, cds, pilot)
    
lab=[]

AC=Pilot.AutoCar()
cam=Camera(300,300)
lidar=LiDAR.Rplidar()

/usr/local/lib/python3.6/dist-packages/numba/errors.py:137: UserWarning: Insufficiently recent colorama version found. Numba requires colorama >= 0.3.9
  warnings.warn(msg)


- 감지된 장치 -
[O] 6축 센서
[O] 휠/카메라 모터 드라이버
[O] 전/후진 모터 드라이버
[O] 오디오 드라이버


In [9]:
cds=Cds(7)
lab.append(widgets.Label(value="Cds : 0"))
display(lab[-1])

dtime = time.time()
while time.time()-dtime<30:
    lab[-1].value="Cds : "+str(cds.read())
    time.sleep(0.1)

Label(value='Cds : 0')

In [10]:
accX=widgets.Label(value="Acc_X : 0")
accY=widgets.Label(value="Acc_Y : 0")
accZ=widgets.Label(value="Acc_Z : 0")
gyroX=widgets.Label(value="Gyro_X : 0")
gyroY=widgets.Label(value="Gyro_Y : 0")
gyroZ=widgets.Label(value="Gyro_Z : 0")

display(accX)
display(accY)
display(accZ)
display(gyroX)
display(gyroY)
display(gyroZ)

lasttime = 0
dtime = time.time()
while time.time()-dtime<30:
    if time.time()-lasttime>1:
        acc=AC.getAccel()
        gyro=AC.getGyro()

        accX.value="Acc_X : "+str(acc['x'])
        accY.value="Acc_Y : "+str(acc['y'])
        accZ.value="Acc_Z : "+str(acc['z'])
        gyroX.value="Gyro_X : "+str(gyro['x'])
        gyroY.value="Gyro_Y : "+str(gyro['y'])
        gyroZ.value="Gyro_Z : "+str(gyro['z'])
        lasttime=time.time()

Label(value='Acc_X : 0')

Label(value='Acc_Y : 0')

Label(value='Acc_Z : 0')

Label(value='Gyro_X : 0')

Label(value='Gyro_Y : 0')

Label(value='Gyro_Z : 0')

In [11]:
lidar.connect()
lidar.startMotor()
lasttime=0
dtime = time.time()
while time.time()-dtime<30:
    if time.time()-lasttime>0.03:
        data=lidar.getMap(size=(300,300))
        Util.imshow("map", data, width=600, height=600)
        lasttime=time.time()
lidar.stopMotor()

Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

In [12]:
AC.camTilt(0)
AC.camPan(90)
AC.steering=0

In [13]:
dtime = time.time()
lasttime=0
sw=True
while time.time()-dtime<30:
    if time.time()-lasttime>2:
        if sw:
            AC.camTilt(0)
            AC.camPan(10)
            AC.steering=-1
            AC.forward(99)
            sw=False
        else:
            AC.camTilt(90)
            AC.camPan(180)
            AC.steering=1
            AC.backward(99)
            sw=True
        lasttime=time.time()
        
AC.stop()

In [14]:
!timeout 30s rec test.mp3 

rec WARN alsa: can't encode 0-bit Unknown or not applicable

Input File     : 'default' (alsa)
Channels       : 2
Sample Rate    : 48000
Precision      : 16-bit
Sample Encoding: 16-bit Signed Integer PCM

In:0.00% 00:00:22.44 [00:00:00.00] Out:1.07M [      |      ]        Clip:0    
Aborted.


In [15]:
!play test.mp3 

play WARN alsa: can't encode 0-bit Unknown or not applicable

test.mp3:

 File Size: 358k      Bit Rate: 128k
  Encoding: MPEG audio    
  Channels: 2 @ 16-bit   
Samplerate: 48000Hz      
Replaygain: off         
  Duration: 00:00:22.39  

In:99.9% 00:00:22.37 [00:00:00.02] Out:1.07M [      |      ]        Clip:0    
Done.


In [20]:
!play 안녕하세요.mp3 

play WARN alsa: can't encode 0-bit Unknown or not applicable

안녕하세요.mp3:

 File Size: 5.09k     Bit Rate: 32.0k
  Encoding: MPEG audio    
  Channels: 1 @ 16-bit   
Samplerate: 24000Hz      
Replaygain: off         
  Duration: 00:00:01.27  

In:98.1% 00:00:01.25 [00:00:00.02] Out:30.0k [ =====|===== ] Hd:5.0 Clip:0    
Done.


In [17]:
LED=Pilot.PWM(1,0x5c)
LED.setFreq(50)

dtime = time.time()
lasttime=0
sw=0
while time.time()-dtime<30:
    if time.time()-lasttime>1:
        if sw==0:
            LED.setDuty(0,99)
            LED.setDuty(1,99)
            LED.setDuty(2,99)
            LED.setDuty(3,99)
        elif sw==2:
            LED.setDuty(4,99)
            LED.setDuty(5,99)
            LED.setDuty(6,99)
            LED.setDuty(7,99)
        else:
            LED.setDuty(0,0)
            LED.setDuty(1,0)
            LED.setDuty(2,0)
            LED.setDuty(3,0)
            LED.setDuty(4,0)
            LED.setDuty(5,0)
            LED.setDuty(6,0)
            LED.setDuty(7,0)
        sw+=1
        sw%=4
        lasttime=time.time()

In [19]:
from pop import PiezoBuzzer

p = PiezoBuzzer(12)

butterfly_scale = [4,4,4, 4,4,4, 4,4,4,4, 4,4,4,  4,4,4,4, 4,4,4, 4,4,4,4, 4,4,4]
butterfly_pitch = [8,5,5, 6,3,3, 1,3,5,6, 8,8,8,  8,5,5,5, 6,3,3, 1,5,8,8, 5,5,5]
butterfly_duration = [8,8,4,   8,8,4, 8,8,8,8, 8,8,4,  8,8,8,8, 8,8,4, 8,8,8,8, 8,8,4]
sheet_butterfly = [butterfly_scale, butterfly_pitch, butterfly_duration]

p.play(sheet_butterfly)

Exception in thread Thread-12:
Traceback (most recent call last):
  File "/usr/lib/python3.6/threading.py", line 916, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.6/threading.py", line 864, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.6/dist-packages/pop/__init__.py", line 100, in __raw_run
    self.run()
  File "/usr/local/lib/python3.6/dist-packages/pop/__init__.py", line 2630, in run
    self.tone(s, p, d)
  File "/usr/local/lib/python3.6/dist-packages/pop/__init__.py", line 2610, in tone
    self.piezo.stop()
  File "/usr/lib/python3/dist-packages/Jetson/GPIO/gpio.py", line 637, in stop
    _disable_pwm(self._ch_info)
  File "/usr/lib/python3/dist-packages/Jetson/GPIO/gpio.py", line 273, in _disable_pwm
    with open(_pwm_enable_path(ch_info), 'w') as f:
FileNotFoundError: [Errno 2] No such file or directory: '/sys/devices/32f0000.pwm/pwm/pwmchip4/pwm0/enable'

